# 04 — Grad-CAM Interpretability Tests

This notebook tests the Grad-CAM module for visual explanations of CNN predictions.

**Goals:**
- Generate and visualize heatmaps on retinal images
- Verify heatmaps highlight clinically relevant regions
- Compare heatmaps across DR grades
- Test with different target classes

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from src.config import load_settings
from src.models.retinal_cnn import build_cnn_model
from src.explainability.grad_cam import (
    generate_grad_cam,
    overlay_heatmap,
    explain_prediction,
    find_target_layer,
)

settings = load_settings()
print('Modules loaded.')

## 1. Setup — Build Model and Create Test Image

In [ ]:
# Build model (with random weights for testing)
model = build_cnn_model(freeze_base=False)
print(f'Model: {model.name}')

# Find the target conv layer
target_layer = find_target_layer(model)
print(f'Target layer for Grad-CAM: {target_layer}')

# Create a synthetic retinal image for testing
target_size = settings['image']['target_size']
test_image = np.random.rand(target_size[1], target_size[0], 3).astype(np.float32)
print(f'Test image shape: {test_image.shape}')

## 2. Generate Grad-CAM Heatmap

In [ ]:
# Get prediction
pred = model.predict(np.expand_dims(test_image, axis=0), verbose=0)[0]
predicted_class = np.argmax(pred)
grade_labels = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative']

print(f'Predicted class: {predicted_class} ({grade_labels[predicted_class]})')
print(f'Confidence: {pred[predicted_class]:.4f}')
print(f'All probabilities: {[f"{p:.3f}" for p in pred]}')

In [ ]:
# Generate heatmap for the predicted class
heatmap = generate_grad_cam(model, test_image, target_class=predicted_class)

print(f'Heatmap shape: {heatmap.shape}')
print(f'Heatmap range: [{heatmap.min():.3f}, {heatmap.max():.3f}]')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Original image
axes[0].imshow(test_image)
axes[0].set_title('Input Image')
axes[0].axis('off')

# Raw heatmap
axes[1].imshow(heatmap, cmap='jet')
axes[1].set_title(f'Grad-CAM Heatmap\n(Class: {grade_labels[predicted_class]})')
axes[1].axis('off')

# Overlay
overlay = overlay_heatmap(test_image, heatmap, alpha=0.4)
axes[2].imshow(overlay)
axes[2].set_title('Overlay (α=0.4)')
axes[2].axis('off')

plt.suptitle('Grad-CAM Visualization', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Compare Heatmaps Across Target Classes

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 8))

for grade in range(5):
    heatmap = generate_grad_cam(model, test_image, target_class=grade)
    overlay = overlay_heatmap(test_image, heatmap, alpha=0.4)
    
    axes[0, grade].imshow(heatmap, cmap='jet')
    axes[0, grade].set_title(f'Grade {grade}: {grade_labels[grade]}')
    axes[0, grade].axis('off')
    
    axes[1, grade].imshow(overlay)
    axes[1, grade].set_title(f'Prob: {pred[grade]:.3f}')
    axes[1, grade].axis('off')

axes[0, 0].set_ylabel('Raw Heatmap', fontsize=12)
axes[1, 0].set_ylabel('Overlay', fontsize=12)
plt.suptitle('Grad-CAM: Explaining Each Class', fontsize=14)
plt.tight_layout()
plt.show()

## 4. Full Explanation Pipeline

In [ ]:
overlay_img, heatmap, pred_class, confidence = explain_prediction(model, test_image)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(test_image)
axes[0].set_title('Input')
axes[0].axis('off')

axes[1].imshow(overlay_img)
axes[1].set_title(f'Prediction: {grade_labels[pred_class]} ({confidence:.1%})')
axes[1].axis('off')

plt.suptitle('explain_prediction() Output', fontsize=14)
plt.tight_layout()
plt.show()

print(f'Predicted grade: {pred_class} ({grade_labels[pred_class]})')
print(f'Confidence: {confidence:.4f}')

## 5. Alpha Blending Comparison

In [ ]:
alphas = [0.1, 0.3, 0.5, 0.7, 0.9]
heatmap = generate_grad_cam(model, test_image)

fig, axes = plt.subplots(1, len(alphas), figsize=(4 * len(alphas), 4))
for i, alpha in enumerate(alphas):
    overlay = overlay_heatmap(test_image, heatmap, alpha=alpha)
    axes[i].imshow(overlay)
    axes[i].set_title(f'α = {alpha}')
    axes[i].axis('off')

plt.suptitle('Heatmap Overlay — Alpha Comparison', fontsize=14)
plt.tight_layout()
plt.show()

## Summary

- Grad-CAM module works end-to-end with the CNN architecture
- Heatmaps are generated for any target class
- Overlay blending produces clear visualizations
- Alpha = 0.4 provides a good balance of visibility
- With real trained weights, heatmaps should highlight:
  - Microaneurysms, hemorrhages (Mild/Moderate DR)
  - Neovascularization, cotton-wool spots (Severe/Proliferative DR)
  - Macula and optic disc regions